In [22]:
import requests
import json
from dotenv import load_dotenv
import os
import re
from datetime import datetime

load_dotenv()
custom_request_header = os.getenv("CUSTOM_REQUEST_HEADER")

In [2]:
USER_API_URL = 'http://localhost:3000/api/users'
HEADERS = { "x-customrequired-header": custom_request_header }

In [6]:
# Get a List of all users
r = requests.get(USER_API_URL, headers=HEADERS)
users = json.loads(r.content)

In [21]:
user_dict = {}
for user in users:
    user_dict[user['_id']] = user

In [15]:
def identify_problem_users(users):
    users_with_capital_emails = [user for user in users if re.compile('[A-Z]').search(user['email'])]

    potential_duplicate_emails = set([user['email'].lower() for user in users_with_capital_emails])

    problem_users = {}
    for user in users:
        current_email = user['email'].lower()
        if current_email in potential_duplicate_emails:
            if problem_users.get(current_email, None) is not None:
                problem_users[current_email].append(user['_id'])
            else:
                problem_users[current_email] = [user['_id']]

    non_duped_capital_emails_with_ids = set([(email, problem_users[email][0]) for email in problem_users.keys() if len(problem_users[email]) == 1])

    for email, user_id in non_duped_capital_emails_with_ids:
        problem_users.pop(email)

    return problem_users, non_duped_capital_emails_with_ids

In [16]:
duped_emails, non_duped_capital_emails_with_ids = identify_problem_users(users)

In [ ]:
duped_emails

In [ ]:
non_duped_capital_emails_with_ids

In [11]:
def update_user_email(email, user_id):
    r = requests.patch(USER_API_URL + '/' + user_id, json={'email': email}, headers=HEADERS)
    print(r.content)
    
def fix_non_duped_capital_emails(emails_with_ids):
    for email, user_id in emails_with_ids:
        print(email, user_id)
        update_user_email(user_id, email)


In [ ]:
users[0]

In [37]:
def determine_canonical_id_for_duped_email(duped_email_ids, user_dict):
    canonical_id = ''
    oldest_creation_date = datetime.now()
    for user_id in duped_email_ids:
        current_creation_date = datetime.strptime(user_dict[user_id]['createdDate'], '%Y-%m-%dT%H:%M:%S.%fZ')
        if current_creation_date < oldest_creation_date:
            oldest_creation_date = current_creation_date
            canonical_id = user_id

    return canonical_id

In [ ]:
determine_canonical_id_for_duped_email(duped_emails['dannyprikaz@gmail.com'], user_dict)

In [ ]:
duped_emails

In [39]:
r = requests.get('http://localhost:3000/api/projectteammembers', headers=HEADERS)

In [ ]:
r.content

# We need to find all of the other documents in the database that reference the duplicated users.

### Database objects that reference User Ids:
- `checkIn`: checkIns have a userId and an eventId, but the API does not expose endpoints for deleting or updating checkIns. We could keep the IDs of every checkIn with a duplicated user and run them through a script that directly accesses the mongooes object, or we could attempt to access Mongo directly in this script
- `event`: events have a field for owner with an ownerId, which is supposed to be "id of user who created event." None of the events in the database have the owner field filled in.
- `project`: projects have a field called `managedByUsers`. For most documents in the database, this is an empty array. For the ones where it is not an empty array, it does not contain any valid userIds.
- `projectTeamMember`: This seems like it should have plenty of data containing userIds, but there are no projectTeamMember documents in the database
- `recurringEvent`: Similar to event, this has a field for owner with ownerId. It is set to default to '123456', and there are no documents in the database that don't have that value.
- 